# Cleaning ACS Data

**Goal:**  
Prepare the ACS data so it can be used later in the housing affordability analysis.

**Plan:**
1. Load the raw ACS files
2. Combine the yearly files into one dataset
3. Check the combined data
4. Rename and organize the columns
5. Fix data types
6. Clean county and FIPS columns
7. Create useful housing variables
8. Save the cleaned ACS dataset

In [15]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

### Step 1: Loading Raw ACS Files

In [17]:
project_root = Path.cwd().parent
acs_path = project_root / "data" / "ACS"
clean_path = project_root / "clean_data"
clean_path.mkdir(exist_ok=True)
acs_files = sorted(acs_path.glob("acs*.csv"))

print("CSV files found:")
for file in acs_files:
    print(file.name)

CSV files found:
acs2005.csv
acs2006.csv
acs2007.csv
acs2008.csv
acs2009.csv
acs2010.csv
acs2011.csv
acs2012.csv
acs2013.csv


### Step 2: Combining the Files

In [20]:
dfs = []

for file in acs_files:
    df = pd.read_csv(file, dtype=str)
    
    year = int(file.stem.replace("acs", ""))
    df["year"] = year
    
    dfs.append(df)

acs = pd.concat(dfs, ignore_index=True)

acs.head()


,NAME,B25007_001E,B25007_002E,B25007_003E,B25007_004E,B25007_005E,B25007_006E,B25007_007E,B25007_008E,B25007_009E,...,B25007_011E,B25007_012E,B25007_013E,B25007_014E,B25007_015E,B25007_016E,B25007_017E,state,county,year
0,"Atlantic County, New Jersey",101584,68970,980,7452,14735,16400,6754,5659,7839,...,1556,32614,3410,6992,7108,6368,1624,34,001,2005
1,"Bergen County, New Jersey",332223,225175,763,13773,50079,54557,24781,20251,28909,...,7542,107048,4393,25468,25357,20069,8337,34,003,2005
2,"Burlington County, New Jersey",164046,129201,1360,14486,28484,34191,11863,10931,14120,...,2271,34845,3498,9908,6761,5952,2318,34,005,2005
3,"Camden County, New Jersey",190659,130576,1141,14690,26698,33575,14054,10884,14414,...,2419,60083,6026,14571,13478,10447,3196,34,007,2005
4,"Cape May County, New Jersey",43616,34101,257,2457,5958,7586,3447,3174,5332,...,1272,9515,1123,2035,1614,1190,680,34,009,2005


In [21]:
acs.shape

(189, 21)

this shows that data is loaded correctly with 21 counties across 9 years.

### Step 3: Checking the Combined Data

In [22]:
# Check data types and non-null counts
acs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189 entries, 0 to 188
Data columns (total 21 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   NAME         189 non-null    object
 1   B25007_001E  189 non-null    object
 2   B25007_002E  189 non-null    object
 3   B25007_003E  189 non-null    object
 4   B25007_004E  189 non-null    object
 5   B25007_005E  189 non-null    object
 6   B25007_006E  189 non-null    object
 7   B25007_007E  189 non-null    object
 8   B25007_008E  189 non-null    object
 9   B25007_009E  189 non-null    object
 10  B25007_010E  189 non-null    object
 11  B25007_011E  189 non-null    object
 12  B25007_012E  189 non-null    object
 13  B25007_013E  189 non-null    object
 14  B25007_014E  189 non-null    object
 15  B25007_015E  189 non-null    object
 16  B25007_016E  189 non-null    object
 17  B25007_017E  189 non-null    object
 18  state        189 non-null    object
 19  county       189 non-null    

In [23]:
# Check column names
acs.columns

Index(['NAME', 'B25007_001E', 'B25007_002E', 'B25007_003E', 'B25007_004E',
       'B25007_005E', 'B25007_006E', 'B25007_007E', 'B25007_008E',
       'B25007_009E', 'B25007_010E', 'B25007_011E', 'B25007_012E',
       'B25007_013E', 'B25007_014E', 'B25007_015E', 'B25007_016E',
       'B25007_017E', 'state', 'county', 'year'],
      dtype='object')

In [24]:
# Check missing values
acs.isnull().sum()

NAME           0
B25007_001E    0
B25007_002E    0
B25007_003E    0
B25007_004E    0
B25007_005E    0
B25007_006E    0
B25007_007E    0
B25007_008E    0
B25007_009E    0
B25007_010E    0
B25007_011E    0
B25007_012E    0
B25007_013E    0
B25007_014E    0
B25007_015E    0
B25007_016E    0
B25007_017E    0
state          0
county         0
year           0
dtype: int64

### Step 4: Renaming Columns

The ACS columns use Census codes, so I renamed them to make the data easier to read and use.

In [25]:
acs = acs.rename(columns={
    "NAME": "county_name",
    "B25007_001E": "total_occupied_units",
    "B25007_002E": "owner_occupied_total",
    "B25007_003E": "owner_occupied_hh_15_24",
    "B25007_004E": "owner_occupied_hh_25_34",
    "B25007_005E": "owner_occupied_hh_35_44",
    "B25007_006E": "owner_occupied_hh_45_54",
    "B25007_007E": "owner_occupied_hh_55_59",
    "B25007_008E": "owner_occupied_hh_60_64",
    "B25007_009E": "owner_occupied_hh_65_74",
    "B25007_010E": "owner_occupied_hh_75_84",
    "B25007_011E": "owner_occupied_hh_85_plus",
    "B25007_012E": "renter_occupied_total",
    "B25007_013E": "renter_occupied_hh_15_24",
    "B25007_014E": "renter_occupied_hh_25_34",
    "B25007_015E": "renter_occupied_hh_35_44",
    "B25007_016E": "renter_occupied_hh_45_54",
    "B25007_017E": "renter_occupied_hh_55_59",
    "state": "state_fips",
    "county": "county_fips"
})

acs.head()

,county_name,total_occupied_units,owner_occupied_total,owner_occupied_hh_15_24,owner_occupied_hh_25_34,owner_occupied_hh_35_44,owner_occupied_hh_45_54,owner_occupied_hh_55_59,owner_occupied_hh_60_64,owner_occupied_hh_65_74,...,owner_occupied_hh_85_plus,renter_occupied_total,renter_occupied_hh_15_24,renter_occupied_hh_25_34,renter_occupied_hh_35_44,renter_occupied_hh_45_54,renter_occupied_hh_55_59,state_fips,county_fips,year
0,"Atlantic County, New Jersey",101584,68970,980,7452,14735,16400,6754,5659,7839,...,1556,32614,3410,6992,7108,6368,1624,34,001,2005
1,"Bergen County, New Jersey",332223,225175,763,13773,50079,54557,24781,20251,28909,...,7542,107048,4393,25468,25357,20069,8337,34,003,2005
2,"Burlington County, New Jersey",164046,129201,1360,14486,28484,34191,11863,10931,14120,...,2271,34845,3498,9908,6761,5952,2318,34,005,2005
3,"Camden County, New Jersey",190659,130576,1141,14690,26698,33575,14054,10884,14414,...,2419,60083,6026,14571,13478,10447,3196,34,007,2005
4,"Cape May County, New Jersey",43616,34101,257,2457,5958,7586,3447,3174,5332,...,1272,9515,1123,2035,1614,1190,680,34,009,2005


Each row is one county-year. Owner and renter columns count housing units by the age of the householder. `hh` means householder.

### Step 5: Fixing Data Types

In [26]:
acs.dtypes

county_name                  object
total_occupied_units         object
owner_occupied_total         object
owner_occupied_hh_15_24      object
owner_occupied_hh_25_34      object
owner_occupied_hh_35_44      object
owner_occupied_hh_45_54      object
owner_occupied_hh_55_59      object
owner_occupied_hh_60_64      object
owner_occupied_hh_65_74      object
owner_occupied_hh_75_84      object
owner_occupied_hh_85_plus    object
renter_occupied_total        object
renter_occupied_hh_15_24     object
renter_occupied_hh_25_34     object
renter_occupied_hh_35_44     object
renter_occupied_hh_45_54     object
renter_occupied_hh_55_59     object
state_fips                   object
county_fips                  object
year                          int64
dtype: object

The count columns need to be numeric so they can be used for calculations later.

In [28]:
count_cols = [
    "total_occupied_units",
    "owner_occupied_total",
    "owner_occupied_hh_15_24",
    "owner_occupied_hh_25_34",
    "owner_occupied_hh_35_44",
    "owner_occupied_hh_45_54",
    "owner_occupied_hh_55_59",
    "owner_occupied_hh_60_64",
    "owner_occupied_hh_65_74",
    "owner_occupied_hh_75_84",
    "owner_occupied_hh_85_plus",
    "renter_occupied_total",
    "renter_occupied_hh_15_24",
    "renter_occupied_hh_25_34",
    "renter_occupied_hh_35_44",
    "renter_occupied_hh_45_54",
    "renter_occupied_hh_55_59"
]

for col in count_cols:
    acs[col] = pd.to_numeric(acs[col])

acs["state_fips"] = acs["state_fips"].astype(str)
acs["county_fips"] = acs["county_fips"].astype(str)

acs.dtypes

county_name                  object
total_occupied_units          int64
owner_occupied_total          int64
owner_occupied_hh_15_24       int64
owner_occupied_hh_25_34       int64
owner_occupied_hh_35_44       int64
owner_occupied_hh_45_54       int64
owner_occupied_hh_55_59       int64
owner_occupied_hh_60_64       int64
owner_occupied_hh_65_74       int64
owner_occupied_hh_75_84       int64
owner_occupied_hh_85_plus     int64
renter_occupied_total         int64
renter_occupied_hh_15_24      int64
renter_occupied_hh_25_34      int64
renter_occupied_hh_35_44      int64
renter_occupied_hh_45_54      int64
renter_occupied_hh_55_59      int64
state_fips                   object
county_fips                  object
year                          int64
dtype: object

### Step 6: Cleaning County and FIPS Columns

This step cleans the county name and creates a full county FIPS code so the ACS data can merge with the other datasets later.

In [30]:
acs["county_clean"] = (
    acs["county_name"]
    .str.replace(" County, New Jersey", "", regex=False)
    .str.strip()
)

acs["county_fips_full"] = acs["state_fips"].str.zfill(2) + acs["county_fips"].str.zfill(3)

acs[["county_name", "county_clean", "state_fips", "county_fips", "county_fips_full", "year"]].head()

,county_name,county_clean,state_fips,county_fips,county_fips_full,year
0,"Atlantic County, New Jersey",Atlantic,34,001,34001,2005
1,"Bergen County, New Jersey",Bergen,34,003,34003,2005
2,"Burlington County, New Jersey",Burlington,34,005,34005,2005
3,"Camden County, New Jersey",Camden,34,007,34007,2005
4,"Cape May County, New Jersey",Cape May,34,009,34009,2005


### Step 7: Creating Useful ACS Variables

This step creates simple rates that are easier to compare across counties and years.

In [33]:
acs["owner_rate"] = acs["owner_occupied_total"] / acs["total_occupied_units"]
acs["renter_rate"] = acs["renter_occupied_total"] / acs["total_occupied_units"]

acs[[
    "county_clean",
    "year",
    "total_occupied_units",
    "owner_occupied_total",
    "renter_occupied_total",
    "owner_rate",
    "renter_rate"
]].head()

,county_clean,year,total_occupied_units,owner_occupied_total,renter_occupied_total,owner_rate,renter_rate
0,Atlantic,2005,101584,68970,32614,0.678946,0.321054
1,Bergen,2005,332223,225175,107048,0.677783,0.322217
2,Burlington,2005,164046,129201,34845,0.787590,0.212410
3,Camden,2005,190659,130576,60083,0.684867,0.315133
4,Cape May,2005,43616,34101,9515,0.781846,0.218154


### Step 8: Saving the Cleaned ACS Data

In [38]:
acs.to_csv(clean_path / "ACS_clean.csv", index=False)